# SoccerTrack v2 — treinamento BAS

Continua o treinamento do classificador temporal a partir de `MyDrive/soccertrack-bas-run/bas_spotter.pt` a partir de features MViT-V1-B pré-treinadas em Kinetics-400. O extrator é **congelado**; somente o spotter temporal é treinado. O spotter é inspirado em T-DEED, não a implementação original. Execute as células em ordem no Colab com GPU ou Jupyter local com CUDA.

Obtenha acesso ao [dataset SoccerTrack v2](https://huggingface.co/datasets/atomscott/soccertrack-v2), aceite as condições e baixe vídeos e anotações da release **v1.2** diretamente para o disco local do ambiente. A extração em vídeos panorâmicos completos é demorada e demanda bastante espaço. No Colab, os dados baixados em `/content` são temporários; o cache de features, o checkpoint e as predições são salvos no Google Drive. Os pesos MViT são baixados pelo torchvision na primeira execução.

A anotação v1.2 usa `position` em milissegundos a partir do primeiro frame do vídeo de cada tempo. Verifique visualmente o alinhamento antes de treinar. O baseline usa um único rótulo por amostra: eventos simultâneos podem perder um dos rótulos.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, random

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

WORKSPACE = Path("/content" if IS_COLAB else Path.cwd()).resolve()
# Dados no disco temporário do Colab; resultados no Google Drive.
DATA_ROOT = Path("/content/soccertrack-v2" if IS_COLAB else "./soccertrack-v2-data").resolve()
OUTPUT_ROOT = Path("/content/drive/MyDrive/soccertrack-bas-run" if IS_COLAB else "./soccertrack-bas-run").resolve()
REPO = WORKSPACE / "SoccerTrack-v2-baseline"
REVISION = "a752d294df2a0e816e827b70f284980698f5a50e"
print("Dados:", DATA_ROOT, "\nResultados:", OUTPUT_ROOT)


## Código e dependências

O clone é fixado em um commit; o checkout é recusado se houver alterações locais. Instale PyTorch e torchvision compatíveis com seu CUDA; no Colab os pacotes normalmente já estão instalados. Reinicie o runtime se alterar versões já importadas.


In [ ]:
if not (REPO / ".git").is_dir():
    if REPO.exists():
        raise RuntimeError(f"{REPO} já existe e não é um clone Git")
    subprocess.run(["git", "clone", "https://github.com/AtomScott/SoccerTrack-v2.git", str(REPO)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("O clone contém alterações locais; use outro diretório.")
subprocess.run(["git", "fetch", "origin", REVISION], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPO, check=True)
# Instale somente as dependências necessárias ao BAS. O runtime deve ter torch e torchvision compatíveis.
subprocess.run([sys.executable, "-m", "pip", "install", "numpy", "opencv-python-headless", "pyyaml", "tqdm", "huggingface_hub[cli]"], check=True)


In [ ]:
import cv2
import numpy as np
import torch
import torchvision
from tqdm.auto import tqdm
from dataclasses import asdict

if not torch.cuda.is_available():
    raise RuntimeError("CUDA não disponível. Selecione GPU no Colab ou configure PyTorch com CUDA.")
# Evita importar o pacote local src do ProjetoPDS em vez do pacote upstream.
if "src" in sys.modules and not str(getattr(sys.modules["src"], "__file__", "")).startswith(str(REPO)):
    raise RuntimeError("Outro pacote 'src' já foi importado; reinicie o kernel e execute este notebook primeiro.")
sys.path.insert(0, str(REPO))
from baselines.bas.train import (FeatureCfg, SpotterCfg, build_mvit_extractor, build_spotter,
    labels_to_clip_targets, make_windows, train_spotter, predict_clip_probs, decode_predictions, write_predictions)
from src.data_utils.soccertrack_v2 import BAS_LABELS, Event
print("GPU:", torch.cuda.get_device_name(0), "Torch:", torch.__version__, "Torchvision:", torchvision.__version__)


## Divisão das partidas

Edite as listas antes do download. A divisão padrão usa todas as dez partidas e pode exceder o espaço disponível no Colab; para experimentar, escolha menos partidas, mantendo treino, validação e teste não vazios e disjuntos.


In [ ]:
TRAIN = ["132877", "118575", "118576", "117092", "118577", "118578", "128058"]
VAL = ["117093"]
TEST = ["128057", "132831"]
SEED = 42
feature_cfg = FeatureCfg()
spot_cfg = SpotterCfg()
# Ajuste conforme a memória disponível. As features usam a mesma transformação do MViT oficial.
FEATURE_BATCH = 4
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
all_ids = TRAIN + VAL + TEST
if not all((TRAIN, VAL, TEST)) or len(all_ids) != len(set(all_ids)):
    raise ValueError("As divisões devem ser não vazias, sem partidas repetidas.")



## Download direto para o ambiente

Aceite primeiro as condições na [página do dataset](https://huggingface.co/datasets/atomscott/soccertrack-v2). Execute a autenticação usando um token de leitura autorizado; não inclua o token no notebook. O download seleciona apenas as anotações BAS e os vídeos das partidas configuradas, na revisão `v1.2`. Para usar arquivos existentes, defina `DOWNLOAD_DATA = False` e confirme que são da mesma revisão.

O dry run mostra arquivos e tamanhos antes da transferência. Compare o tamanho total com o espaço disponível; o dataset completo é grande e o Colab pode não ter espaço suficiente. Selecione menos partidas ou use armazenamento com capacidade suficiente. Execute a célula seguinte para iniciar o download.


In [ ]:
DOWNLOAD_DATA = True
DATASET_REVISION = "v1.2"
if DOWNLOAD_DATA:
    import shutil
    from huggingface_hub import login
    # Abre um campo de token no notebook; o token não deve ser salvo nas células.
    login(add_to_git_credential=False)
    hf = shutil.which("hf")
    if hf is None:
        raise RuntimeError("CLI hf não encontrada; execute novamente a célula de dependências.")
    patterns = [pattern for mid in all_ids for pattern in (f"bas/{mid}/*", f"videos/{mid}/*")]
    download_command = [hf, "download", "atomscott/soccertrack-v2", "--type", "dataset",
                        "--revision", DATASET_REVISION, "--local-dir", str(DATA_ROOT),
                        "--include", *patterns]
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DATA_ROOT).free / 1024**3
    print(f"Espaço livre no disco: {free:.1f} GiB")
    subprocess.run(download_command + ["--dry-run"], check=True)


In [ ]:
if DOWNLOAD_DATA:
    # Verifique o tamanho acima antes de executar esta célula.
    subprocess.run(download_command, check=True)
    print("Download concluído em", DATA_ROOT)
else:
    print("Usando dados existentes em", DATA_ROOT)


## Verificação dos dados

São exigidos um JSON BAS e dois vídeos por partida. Os nomes panorâmicos da distribuição são reconhecidos; ajuste `video_path` se a cópia local usar outros nomes. Confirme que as anotações v1.2 e os vídeos estão alinhados.


In [ ]:
def video_path(mid, half):
    # Atualize esta lista caso a cópia local use outros nomes.
    names = ([f"{mid}_panorama_1st_half.mp4", f"{mid}_1st.mp4", f"{mid}_1.mp4", "1.mp4", "1st_half.mp4"] if half == 1 else
             [f"{mid}_panorama_2nd_half.mp4", f"{mid}_2nd.mp4", f"{mid}_2.mp4", "2.mp4", "2nd_half.mp4"])
    matches = [DATA_ROOT / "videos" / mid / name for name in names if (DATA_ROOT / "videos" / mid / name).is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(f"Esperado exatamente um vídeo para {mid}, tempo {half}; encontrados: {matches}. Ajuste video_path se necessário.")
    return matches[0]

def annotation_path(mid):
    return DATA_ROOT / "bas" / mid / f"{mid}_12_class_events.json"

for mid in all_ids:
    if not annotation_path(mid).is_file():
        raise FileNotFoundError(annotation_path(mid))
    for half in (1, 2):
        print(mid, half, video_path(mid, half))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## Extração de features

Os clipes são processados em lotes. O cache novo usa SHA-256 do vídeo e configurações do extrator, permitindo reutilização após novo download do mesmo arquivo. Calcular o hash lê o vídeo inteiro. Caches antigos são reutilizados somente se caminho, tamanho e data ainda coincidirem; caso contrário são extraídos novamente, pois o conteúdo antigo não pode ser verificado com segurança.


In [ ]:
extractor, transform = build_mvit_extractor(feature_cfg, pretrained=True)
extractor = extractor.to("cuda").eval()

def features_for(mid, half):
    path = video_path(mid, half)
    import hashlib
    stat = path.stat()
    cache = OUTPUT_ROOT / "features" / f"{mid}_{half}.npz"
    cache.parent.mkdir(parents=True, exist_ok=True)
    # O cache legado não tem hash: só é seguro reaproveitá-lo com metadados exatos.
    legacy = {"revision": REVISION, "video": str(path), "size": stat.st_size,
              "mtime_ns": stat.st_mtime_ns, "features": asdict(feature_cfg)}
    if cache.exists():
        with np.load(cache, allow_pickle=False) as data:
            old_meta = json.loads(str(data["meta"].item()))
            if old_meta == legacy:
                print(f"Reutilizando cache legado de {mid}, tempo {half}")
                return data["features"], data["centers"]
    digest = hashlib.sha256()
    with path.open("rb") as video_file:
        for chunk in iter(lambda: video_file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    meta = {"revision": REVISION, "video_sha256": digest.hexdigest(),
            "features": asdict(feature_cfg)}
    if cache.exists():
        with np.load(cache, allow_pickle=False) as data:
            if json.loads(str(data["meta"].item())) == meta:
                print(f"Reutilizando cache verificado de {mid}, tempo {half}")
                return data["features"], data["centers"]
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or count <= 0:
        cap.release(); raise ValueError(f"FPS ou número de frames inválido: {path}")
    nclips = max(0, int(np.floor((count / fps - feature_cfg.clip_s) / feature_cfg.stride_s)) + 1)
    if not nclips:
        cap.release(); raise ValueError(f"Vídeo curto demais: {path}")
    # Apenas os frames necessários são decodificados; os índices podem ser compartilhados por clipes.
    span = int(round(feature_cfg.clip_s * fps))
    starts = np.rint(np.arange(nclips) * feature_cfg.stride_s * fps).astype(int)
    indices = [np.linspace(s, s + span - 1, feature_cfg.num_frames).round().astype(int) for s in starts]
    centers = np.rint((np.arange(nclips) * feature_cfg.stride_s + feature_cfg.clip_s / 2) * 1000).astype(np.int64)
    # Map each requested frame to all clips using it; overlapping clips are supported.
    requested = {}
    for clip_id, frame_ids in enumerate(indices):
        for frame_id in set(map(int, frame_ids)):
            requested.setdefault(frame_id, []).append(clip_id)
    result, batch, active = [], [], {}
    next_clip = 0
    try:
        with torch.no_grad():
            for frame_idx in tqdm(range(int(indices[-1][-1]) + 1), desc=f"{mid} tempo {half}"):
                ok, frame = cap.read()
                if not ok:
                    raise RuntimeError(f"Falha lendo frame {frame_idx} de {path}")
                if frame_idx not in requested:
                    continue
                rgb = torch.from_numpy(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)
                for clip_id in requested[frame_idx]:
                    active.setdefault(clip_id, {})[frame_idx] = rgb
                while next_clip < nclips and frame_idx == indices[next_clip][-1]:
                    frames = active.pop(next_clip)
                    clip = torch.stack([frames[int(i)] for i in indices[next_clip]])
                    batch.append(transform(clip))
                    next_clip += 1
                    if len(batch) == FEATURE_BATCH or next_clip == nclips:
                        result.append(extractor(torch.stack(batch).to("cuda")).cpu().numpy())
                        batch.clear()
    finally:
        cap.release()
    if next_clip != nclips:
        raise RuntimeError(f"Leitura incompleta: {next_clip}/{nclips} clipes")
    features = np.concatenate(result).astype(np.float32)
    np.savez_compressed(cache, features=features, centers=centers, meta=json.dumps(meta, sort_keys=True))
    return features, centers


## Preparação e treinamento

O parser usa a lista `actions` da release v1.2 e `position` em milissegundos desde o primeiro quadro de cada tempo. Rótulos são normalizados sem distinguir maiúsculas; classes desconhecidas geram erro. O baseline atribui somente uma classe por clipe.

Configure `TRAIN`, `VAL` e `TEST` acima de acordo com seu experimento. A divisão padrão considera que 132877, 118575 e 118576 já foram usadas no treinamento informado. A célula seguinte carrega o checkpoint existente e continua o treinamento.


In [ ]:
def events_for(mid):
    path = annotation_path(mid)
    raw = json.loads(path.read_text())
    if not isinstance(raw, dict):
        raise ValueError(f"Formato BAS inválido: {path}")
    if "match_id" in raw and str(raw["match_id"]) != str(mid):
        raise ValueError(f"ID da partida incompatível em {path}: {raw['match_id']}")
    items = raw.get("actions", raw.get("annotations"))
    if not isinstance(items, list):
        raise ValueError(f"Lista actions/annotations ausente em {path}; chaves: {list(raw)}")
    labels = {label.casefold(): label for label in BAS_LABELS}
    events = []
    for item in items:
        try:
            half = int(item["gameTime"].split(" - ")[0])
            t_ms = int(item["position"])
            label = labels[item["label"].strip().casefold()]
        except (KeyError, ValueError, TypeError, AttributeError) as exc:
            raise ValueError(f"Anotação inválida em {path}: {item}") from exc
        if half not in (1, 2) or t_ms < 0:
            raise ValueError(f"Tempo inválido em {path}: {item}")
        events.append(Event(half=half, clock="", t_ms=t_ms, label=label))
    return events

def training_windows(match_ids):
    windows, targets = [], []
    for mid in match_ids:
        events = events_for(mid)
        for half in (1, 2):
            feats, centers = features_for(mid, half)
            y = labels_to_clip_targets(events, centers, half)
            xw, yw = make_windows(feats, y, spot_cfg.window)
            windows.append(xw); targets.append(yw)
            print(mid, half, "clipes:", len(feats), "positivos:", int((y > 0).sum()))
    return np.concatenate(windows), np.concatenate(targets)

def load_checkpoint(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint.get("revision") != REVISION or checkpoint.get("labels") != list(BAS_LABELS):
        raise ValueError("Checkpoint incompatível com a revisão ou classes do baseline.")
    if checkpoint.get("feature_cfg") != asdict(feature_cfg):
        raise ValueError("Configuração de features difere do checkpoint. Configure-a antes de criar o extrator.")
    if checkpoint.get("dataset_release") != "v1.2":
        raise ValueError("Versão do dataset incompatível com v1.2.")
    return checkpoint

def assert_held_out(previous_train, candidates):
    overlap = set(map(str, previous_train)) & set(map(str, candidates))
    if overlap:
        raise ValueError(f"Partidas usadas no treinamento também estão na avaliação: {sorted(overlap)}")


## Continuar treinamento existente

Execute as células anteriores em ordem. O checkpoint original deve estar disponível no Drive no caminho indicado abaixo. Informe em `PREVIOUS_TRAIN` **todas** as partidas realmente usadas anteriormente: metadados de checkpoints antigos podem refletir uma lista diferente da execução. Todas devem estar em `TRAIN`, fora de `VAL` e `TEST`. A continuação reutiliza os pesos e reinicia o AdamW, pois o checkpoint original não inclui o estado do otimizador. O arquivo de saída deve ser novo.


In [ ]:
CHECKPOINT_IN = OUTPUT_ROOT / "bas_spotter.pt"
CHECKPOINT_OUT = OUTPUT_ROOT / "bas_spotter_continued.pt"
PREVIOUS_TRAIN = ["132877", "118575", "118576"]  # Confirme conforme o histórico real.
CONTINUATION_EPOCHS = 20
if CHECKPOINT_OUT.exists():
    raise FileExistsError(f"Escolha outro nome; não será sobrescrito: {CHECKPOINT_OUT}")
checkpoint = load_checkpoint(CHECKPOINT_IN)
if not PREVIOUS_TRAIN:
    raise ValueError("Informe as partidas realmente usadas antes deste checkpoint.")
previous = set(map(str, PREVIOUS_TRAIN)) | set(map(str, checkpoint.get("train", [])))
assert_held_out(previous | set(TRAIN), VAL + TEST)
if not previous <= set(TRAIN):
    raise ValueError(f"Inclua as partidas anteriores em TRAIN: {sorted(previous - set(TRAIN))}")
spot_cfg = SpotterCfg(**checkpoint["spotter_cfg"])
spot_cfg.epochs = CONTINUATION_EPOCHS
spotter = build_spotter(feature_cfg, spot_cfg)
spotter.load_state_dict(checkpoint["state_dict"])
x, y = training_windows(TRAIN)
spotter = train_spotter(spotter, x, y, spot_cfg, device="cuda", verbose=True)
continued = {**checkpoint, "state_dict": spotter.cpu().state_dict(),
             "spotter_cfg": asdict(spot_cfg), "train": list(TRAIN),
             "val": list(VAL), "test": list(TEST),
             "previous_train": sorted(previous), "continued_from": str(CHECKPOINT_IN)}
torch.save(continued, CHECKPOINT_OUT)
print("Checkpoint continuado:", CHECKPOINT_OUT)
del x, y


## Predições de validação e teste

Selecione explicitamente o checkpoint abaixo; a célula carrega os pesos escolhidos e verifica se as partidas de avaliação foram usadas no treinamento. Os arquivos de previsões existentes das mesmas partidas serão substituídos. Use validação para ajustar limiar e NMS antes de olhar o teste. O avaliador upstream dessa revisão ordena predições por tempo em vez de confiança, então seu mAP **não é confiável**. Exporte as predições e use um avaliador BAS validado e ordenado por confiança antes de publicar métricas.


In [ ]:
# Escolha explicitamente os pesos e as partidas a avaliar.
PREDICTION_CHECKPOINT = OUTPUT_ROOT / "bas_spotter_continued.pt"
PREDICT_MATCHES = VAL  # Use TEST somente após escolher os parâmetros na validação.
# Informe também partidas anteriores omitidas por metadados antigos.
KNOWN_TRAIN = ["132877", "118575", "118576"]
checkpoint = load_checkpoint(PREDICTION_CHECKPOINT)
assert_held_out(set(map(str, checkpoint.get("train", []))) | set(map(str, KNOWN_TRAIN)), PREDICT_MATCHES)
spot_cfg = SpotterCfg(**checkpoint["spotter_cfg"])
spotter = build_spotter(feature_cfg, spot_cfg)
spotter.load_state_dict(checkpoint["state_dict"])
for mid in PREDICT_MATCHES:
    predicted = []
    for half in (1, 2):
        feats, centers = features_for(mid, half)
        probs = predict_clip_probs(spotter, feats, device="cuda")
        predicted.extend(decode_predictions(probs, centers, half,
            nms_window_s=spot_cfg.nms_window_s, min_confidence=spot_cfg.min_confidence,
            require_above_background=spot_cfg.require_above_background,
            max_peaks_per_class=spot_cfg.max_peaks_per_class))
    path = write_predictions(predicted, OUTPUT_ROOT / "predictions", mid)
    print(mid, len(predicted), path)
